Data Analyst: Andrei Berner Arroyo Tanta

**Project with Numpy and Pandas 2: Institutional Fraud & Anomaly Detection Engine

**Business Application

-Identifying fraudulent transactions in real-time is critical to minimizing financial loss and maintaining customer trust.

-Beneficiaries: Fraud Prevention Units, Internal Audit and Compliance Departments.

-Decision Support: By flagging "mathematically significant" deviations, the system allows investigators to prioritize high-risk cases rather than manually reviewing thousands of standard transactions.

**Executive Summary

This project implements a statistical anomaly detection system using the Median Absolute Deviation (MAD) method, which is more robust to outliers than standard Z-score modeling. It processes 368 complex transactions to isolate high-risk behavior.

In [4]:
#Data Ingestion & Memory Optimization
import numpy as np
import pandas as pd

In [5]:
#Load synthetic fraud dataset
df = pd.read_csv("bills_fraud.xls")
df_work = df.copy()

#Optimization: Categorical and Downscaling types
#Numeric types optimized to int32/float32 to save memory
df_work["step"] = pd.to_numeric(df_work["step"], errors="coerce").astype("int32")
df_work["age"] = pd.to_numeric(df_work["age"], errors="coerce").astype("int32")
df_work["amount"] = pd.to_numeric(df_work["amount"], errors="coerce").astype("float32")
df_work["fraud"] = pd.to_numeric(df_work["fraud"], errors="coerce").astype("float32")

#String columns for better semantic handling
string_cols = ["customer", "gender", "zipcodeOri", "merchant", "zipMerchant", "category"]
df_work[string_cols] = df_work[string_cols].astype("string")

print(f"Dataset optimized. Shape: {df_work.shape}")
df_work.info()

Dataset optimized. Shape: (368, 10)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 368 entries, 0 to 367
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   step         368 non-null    int32  
 1   customer     368 non-null    string 
 2   age          368 non-null    int32  
 3   gender       368 non-null    string 
 4   zipcodeOri   368 non-null    string 
 5   merchant     368 non-null    string 
 6   zipMerchant  368 non-null    string 
 7   category     368 non-null    string 
 8   amount       368 non-null    float32
 9   fraud        368 non-null    float32
dtypes: float32(2), int32(2), string(6)
memory usage: 23.1 KB


In [10]:
#Statistical Engine (NumPy Layer)

#Convert to NumPy for high-speed computation
amounts = df_work["amount"].to_numpy()

#Calculate Median and MAD
median_val = np.median(amounts)
abs_deviation = np.abs(amounts - median_val)
mad_val = 1.4826 * np.median(abs_deviation)

#Generate MAD Scores
df_work["mad_score"] = (amounts - median_val) / mad_val
df_work["is_anomaly"] = np.abs(df_work["mad_score"]) > 3.5 # Standard threshold

print(f"--- Report ---")
print(f"Anomalies detected: {df_work["is_anomaly"].sum()}")

--- Report ---
Anomalies detected: 114


In [7]:
#Data Interrogation (Pandas Layer)

#Define conditions for risk levels
conditions = [
    (df_work['mad_score'] > 10),     # Critical
    (df_work['mad_score'] > 5),      # High
    (df_work['is_anomaly'] == True)  # Moderate
]
labels = ['Critical', 'High', 'Moderate']

df_work['risk_level'] = np.select(conditions, labels, default='Normal')

In [8]:
#Deep Dive: Anomalous Row Extraction
#Extract anomalies for deep interrogation
df_anomalies = df_work.loc[df_work['is_anomaly']]

#Reorganizing columns for an Investigator's View
investigator_cols = ["customer", "risk_level", "category", "merchant", "amount", "mad_score"]
df_report = df_anomalies.reindex(columns=investigator_cols)

df_report.sort_values(by="mad_score", ascending=False).head(10)

,customer,risk_level,category,merchant,amount,mad_score
121,CUST-10013,Critical,food,MERCH-432,15000.987305,29.395103
10,CUST-1839,Critical,food,MERCH-411,15000.123047,29.393362
29,CUST-2121,Critical,clothing,MERCH-432,14000.987305,27.379099
162,CUST-10054,Critical,clothing,MERCH-321,14000.987305,27.379099
173,CUST-10065,Critical,electronics,MERCH-210,14000.987305,27.379099
41,CUST-3333,Critical,clothing,MERCH-987,13500.987305,26.371096
105,CUST-9797,Critical,electronics,MERCH-654,13000.987305,25.363092
47,CUST-3939,Critical,clothing,MERCH-654,13000.987305,25.363092
226,CUST-10118,Critical,electronics,MERCH-987,12500.987305,24.355091
247,CUST-10139,Critical,food,MERCH-987,12500.987305,24.355091


In [9]:
#Business Insights & Summary

#Risk Analysis by Transaction Category
risk_summary = df_work.groupby('category').agg({
    'is_anomaly': 'sum',
    'amount': ['mean', 'max']
}).rename(columns={'sum': 'total_anomalies'})

#Highlighting results
print("--- Category Risk Profile ---")
print(risk_summary)

--- Category Risk Profile ---
                 is_anomaly       amount              
            total_anomalies         mean           max
category                                              
clothing                 32  3027.915771  14000.987305
electronics              47  3567.180664  14000.987305
food                     35  3087.274414  15000.987305
